# Import Libraries

In [1]:
# Import necessary libraries
import sqlite3
from sqlite3 import Error

import pandas as pd
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

import kagglehub
import csv
import os

# Import the Dataset into the Database

In [2]:
# Download the dataset
path = kagglehub.dataset_download("alessandrasala79/ai-vs-human-generated-dataset")

# Connect to the SQLite database
conn = sqlite3.connect('image.db')

# Drop the test_images table if it exists
conn.execute(
                '''
                    DROP TABLE IF EXISTS test_images
                '''
)

# Drop the train_images table if it exists
conn.execute(
                '''
                    DROP TABLE IF EXISTS train_images
                '''
)

# Create the images table
conn.execute(
                '''
                    CREATE TABLE IF NOT EXISTS train_images(
                        image_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
                        image_name TEXT NOT NULL,
                        image_label INTEGER NOT NULL
                    )
                '''
            )

# Create the test_images table
conn.execute(
                '''
                    CREATE TABLE IF NOT EXISTS test_images(
                        image_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
                        image_name TEXT NOT NULL
                    )
                '''
            )

# Import the dataset into the database and create a list of labels
training_labels = []
# Import the training images
train_path = os.path.join(path, 'train.csv')
with open(train_path, 'r') as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header row
    for row in reader:
        image_name = row[1]
        image_label = row[2]
        training_labels.append(image_label)
        if not image_name or not image_label: # Skip if image_name or image_label is empty
            continue
        conn.execute(
            '''
                INSERT INTO train_images (image_name, image_label)
                VALUES (?, ?)
            ''',
            (image_name, image_label)
        )
        
# Import the test images
test_path = os.path.join(path, 'test.csv')
with open(test_path, 'r') as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header row
    for row in reader:
        if not image_name: # Skip if image_name is empty
            continue
        image_name = row[0]
        conn.execute(
            '''
                INSERT INTO test_images (image_name)
                VALUES (?)
            ''',
            (image_name,)
        )

# Commit the changes and close the connection
conn.commit()
conn.close()

# Create Data Pipelines

In [21]:
# Set the GPU memory growth to avoid OOM errors
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
    
# Setup data pipeline
dataframe = pd.read_csv(train_path)
file_paths = dataframe['file_name'].values
labels = dataframe['label'].values
data = tf.data.Dataset.from_tensor_slices((file_paths, labels))

def read_image(image_file, label):
    image = tf.io.read_file(image_file)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment(image, label):
    

b'train_data/a6dcb93f596a43249135678dfcfc17ea.jpg'


# Define Model

In [22]:
model = keras.Sequential

# Compile Model

# Train Model

# Test with Test Data

# Predictions

In [ ]:
# Add Gui

# Reporting and Visualizations